# Build the agent

[Module 04](README.md) · [Workshop home](../README.md) · [Endpoint reference](../docs/inference.md)

**25 minutes.** The capstone. Everything from the earlier modules in one program.

| Piece | From |
|---|---|
| The tool loop | Notebook 1 of this module |
| Document search as a tool | Module 02 |
| A typed final answer | Module 03 |
| `qwen3-8-27b` | The model table in module 01 |

The change in thinking from module 02 is worth naming. There, every question got
documents stuffed into its prompt whether it needed them or not. Here the model
*decides* when to search. That costs fewer tokens, handles questions that need no
search at all, and lets the model search more than once if the first attempt is
unhelpful.

## Step 1: Connect and build the index

In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

root = Path.cwd()
if not (root / ".env").is_file() and (root.parent / ".env").is_file():
    root = root.parent
if not (root / ".env").is_file():
    raise RuntimeError("No .env found. See docs/00_prework.md")

load_dotenv(root / ".env", override=True)
client = OpenAI(
    base_url=os.environ["AKAMAI_BASE_URL"].strip().rstrip("/"),
    api_key=os.environ["AKAMAI_API_KEY"].strip(),
    timeout=120.0,
)

# qwen3-8-27b for every tool exercise. gemma-4-26b-a4b-it fails at this, see the
# README in this folder for the numbers.
MODEL = "qwen3-8-27b"
ROOT_DIR = root
print("Ready.")

In [ ]:
import sys
sys.path.insert(0, str(ROOT_DIR))

from workshop_utils.retrieval import build_index, search as search_index

# Same code you wrote in module 02, imported rather than pasted.
CORPUS = ROOT_DIR / "02_Embeddings_and_RAG" / "assets" / "sample-docs"
index = build_index(client, CORPUS)
print(f"index ready: {len(index)} chunks from {CORPUS.name}/")

## Step 2: Retrieval as a tool

The same search function, wrapped so the model can call it. The description is the
only thing the model knows about it, so it has to say what the corpus contains.
Otherwise the model cannot judge when searching is worthwhile.

In [ ]:
def search_documents(query: str) -> dict:
    hits = search_index(client, index, query, top_k=3)
    return {
        "results": [
            {"source": h["source"], "score": h["score"], "text": h["text"][:600]}
            for h in hits
        ]
    }

AGENT_TOOLS = [{
    "type": "function",
    "function": {
        "name": "search_documents",
        "description": (
            "Search the Akamai Cloud Inference documentation. Covers the available "
            "models and their context limits, rate limits and error codes, tool "
            "calling behaviour, and embeddings. Use it for any factual question "
            "about the platform."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "What to search for"}
            },
            "required": ["query"],
        },
    },
}]

probe = search_documents("embedding dimensions")
print(f"{len(probe['results'])} hits, top from {probe['results'][0]['source']}")

## Step 3: The loop, with a trace

Same loop as notebook 1, with one addition: it prints what it is doing. A trace is
the only practical way to debug an agent. Without one you are guessing why it
answered the way it did.

Note what is **not** here. No `response_format`. Step 5 explains why that matters.

In [ ]:
SYSTEM = (
    "You answer questions about Akamai Cloud Inference. "
    "Search the documentation before answering any factual question. "
    "Answer only from what the search returns, and name the source file. "
    "If the documents do not contain the answer, say so plainly instead of guessing."
)

FUNCTIONS = {"search_documents": search_documents}

def agent(question, max_turns=5):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    print(f"Q: {question}\n")

    for turn in range(1, max_turns + 1):
        reply = client.chat.completions.create(
            model=MODEL, messages=messages, tools=AGENT_TOOLS,
            max_tokens=600, temperature=0.0,
        )
        msg = reply.choices[0].message

        if not msg.tool_calls:
            print(f"  [turn {turn}] answered\n")
            messages.append(msg)
            return msg.content, messages

        messages.append(msg)
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments)
            print(f"  [turn {turn}] {call.function.name}({args})")
            result = FUNCTIONS[call.function.name](**args)
            for hit in result["results"]:
                print(f"            {hit['score']:>6}  {hit['source']}")
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result),
            })

    print(f"  gave up after {max_turns} turns\n")
    return None, messages

text, history = agent("How many dimensions does the embedding model return, and can I ask for fewer?")
print(text)

Read the trace rather than just the answer. You can see the question the model chose
to search for, which sources came back and how strongly they scored, and that the
answer quotes them. When an agent goes wrong, one of those lines is where it went
wrong, and almost always it is the search query.

The answer should give 4,096 and explain that `dimensions` is rejected because the
model has no Matryoshka representation. Neither fact was in the prompt.

## Step 4: A typed answer

Free text is fine for a human. If something downstream consumes this, you want a
schema. Module 03 showed how, and here is the version that works alongside tools.

In [ ]:
from pydantic import BaseModel, Field

class Answer(BaseModel):
    answer: str = Field(description="the answer, two sentences at most")
    answer_found: bool = Field(
        description="true only if the documents contained the answer, false if they did not"
    )
    sources: list[str] = Field(description="source filenames actually used, empty if none")

ANSWER_SCHEMA = {
    "type": "json_schema",
    "json_schema": {"name": "answer", "schema": Answer.model_json_schema()},
}
COMPACT = "Reply with compact JSON on a single line. Do not add whitespace or newlines."

def agent_typed(question, max_turns=5):
    text, history = agent(question, max_turns=max_turns)

    # Second phase. The tool results are already in `history`, so the model has
    # everything it needs and no longer has any reason to call a tool.
    reply = client.chat.completions.create(
        model=MODEL,
        messages=history + [{
            "role": "user",
            "content": (
                "Now format your answer as JSON. Set answer_found to false if the "
                f"documents did not contain the answer. {COMPACT}"
            ),
        }],
        response_format=ANSWER_SCHEMA,
        max_tokens=500,
        temperature=0.0,
    )
    if reply.choices[0].finish_reason != "stop":
        raise RuntimeError(f"formatting call did not finish: {reply.choices[0].finish_reason}")
    return Answer.model_validate_json(reply.choices[0].message.content)

result = agent_typed("Which model should I use for tool calling, and which should I avoid?")
print(result.model_dump_json(indent=2))

A validated Python object carrying the answer, whether the documents actually
contained it, and which files were used. That is something you can return from an API.

Two phases, deliberately. The loop runs with no `response_format`, and the schema
goes on a separate request afterwards. Step 5 shows what happens if you try to do both
at once, and it is worth running.

## Step 5: Why two phases

This is the one failure in this workshop you are most likely to hit in your own code,
so it is worth seeing rather than being told.

The cell below does the obvious thing: attaches the tool and the schema to the same
request, and asks a question whose answer is only available by searching.

In [ ]:
broken = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": "How many dimensions does the embedding model return?"},
    ],
    tools=AGENT_TOOLS,
    response_format=ANSWER_SCHEMA,
    max_tokens=250,
    temperature=0.0,
)

ch = broken.choices[0]
content = ch.message.content or ""
whitespace = sum(1 for c in content if c in " \n\t")

print("finish_reason :", ch.finish_reason)
print("tool_calls    :", [c.function.name for c in (ch.message.tool_calls or [])] or None)
print("output tokens :", broken.usage.completion_tokens)
print(f"content       : {len(content)} chars, {whitespace} whitespace")
print(f"\n{content[:220]!r}")

**`tool_calls` is `None`.** The model never searched. That is the whole problem, and
everything else follows from it.

`response_format` constrains every token the model may emit to tokens that fit the
schema. A tool call is not part of the schema, so the model *cannot* ask to search.
But the number it needs only exists behind the search. It has no legal move that makes
progress, and you will see one of two failures depending on the schema and the
question:

- It emits whitespace, which JSON permits anywhere, until `max_tokens` runs out.
  `finish_reason` is `length` and you are billed for the whole budget.
- It fills the schema in with invented values and stops cleanly.

The second is far more dangerous, because it looks like success. Let us check whether
that is what happened here.

In [ ]:
real_files = sorted(p.name for p in CORPUS.glob("*.md"))

if ch.finish_reason == "length":
    print("This run hit the whitespace stall. Budget spent, nothing usable.")
else:
    fabricated = Answer.model_validate_json(content)
    invented = [s for s in fabricated.sources if s not in real_files]
    print("It answered cleanly and confidently:")
    print(f"   answer       : {fabricated.answer}")
    print(f"   answer_found : {fabricated.answer_found}")
    print(f"   sources      : {fabricated.sources}")
    print()
    print(f"files that actually exist : {real_files}")
    print(f"files it invented         : {invented or 'none'}")
    print()
    print("The real answer is 4,096, which the agent in Step 3 found by searching.")

If you got the second failure, look at what it produced. A plausible number, a
confident `answer_found: true`, and a citation to a file that does not exist. No
error, no warning, nothing in `finish_reason` to catch. It is exactly the output
shape you were about to trust.

This is also a different problem from the stall in module 03. That one was the model
choosing whitespace out of uncertainty, and a "reply compactly" instruction fixed it.
This one is structural. No prompt can fix it, because the move the model needs is
forbidden by the grammar it has been given.

**Two ways out.**

Run the phases separately, which is what `agent_typed` does. The loop finishes with no
schema, then one more request adds the schema once the answers are in the
conversation. Tools can stay attached on that final request, because the model has no
reason left to call them.

Or set `tool_choice="required"` on the first request, which forces the tool call and
takes the decision away from the model.

The rule to remember: **never set `response_format` on a request where the model might
still need to call a tool.**

## Step 6: The question with no answer

An agent that always produces an answer is worse than useless, because you cannot
tell its guesses from its knowledge. The corpus says nothing about pricing.

In [ ]:
result = agent_typed("What does Akamai Cloud Inference charge per million tokens?")
print(result.model_dump_json(indent=2))

It searched, got nothing relevant, and said so. `answer_found` comes back `false`,
and that is the field you would branch on in real code: escalate to a human, try
another source, or tell the user you do not know.

Be clear about what produced that refusal, because it is not what people assume. It
was not the similarity scores. Look at the trace: the top score is around 0.75, which
is not much lower than the scores on questions it answered well. Similarity search
always returns its best three chunks no matter how irrelevant they are, so a score
threshold would not have caught this.

The refusal came from one sentence in `SYSTEM`, the one telling it to say so plainly
instead of guessing. Delete that sentence and the model fills the gap, exactly as it
did in Step 5.

## Validation

In [ ]:
good = agent_typed("What is the context limit of qwen3-8-27b?")
assert "32,768" in good.answer or "32768" in good.answer, \
    f"expected the real limit in the answer, got {good.answer!r}"
assert good.answer_found is True, "the limit is in the documents, so answer_found should be True"
assert good.sources, "the typed answer carried no sources"

# The schema-plus-tools request must not have searched. Either failure is acceptable,
# the point is that it could not call the tool.
assert not (broken.choices[0].message.tool_calls or []), \
    "with response_format set, the model should not have been able to call a tool"

assert isinstance(result.answer_found, bool), "answer_found should be a boolean"
print("\nAgent complete.")
print("It searches when it needs to, returns a typed answer, and refuses when it should.")

## Clean up

The index is in memory and goes away with the kernel. Nothing was written to disk and
nothing was created in your cloud account.

## You are done

Four modules. You can call the endpoint, build retrieval over your own documents, use
image input and reasoning and typed output, and write an agent loop that you can
debug.

Two things to take with you:

- Check `finish_reason` before using any response.
- Never set `response_format` on a request where a tool call might still be needed.

Optional next step: **[deploy](../deploy/)** puts this agent behind an HTTP endpoint
on a Linode, with the key on the server instead of in a notebook. It creates billable
resources and states what they cost before creating them.

Reference: [endpoint details](../docs/inference.md) · [troubleshooting](../docs/troubleshooting.md)